In [1]:
import os
os.chdir("../..")

nmea_sentences_active_1 = {
    "GPVTG": '$GPVTG,,,,,,,,,N*30 $GPGGA,202233.00,,,,,0,00,99.99,,,,,,*64',
    "GPGSA": '$GPGSA,A,3,30,20,13,14,11,,,,,,,,4.17,3.14,2.74*01',
    "GPGSV_1": '$GPGSV,4,1,14,05,68,261,,07,37,058,,08,02,052,,09,02,096,*70',
    "GPGSV_2": '$GPGSV,4,2,14,11,04,201,,13,57,276,,14,28,138,29,15,21,283,*7E',
    "GPGSV_3": '$GPGSV,4,3,14,18,18,322,,20,53,190,28,21,56,171,,22,15,152,*7B',
    "GPGSV_4":'$GPGSV,4,4,14,27,04,019,,30,69,085,*74',
    "GPGLL": '$GPGLL,,,,,202233.00,V,N*48',
    "GPRMC": '$GPRMC,200517.00,A,5211.77084,N,00213.26972,W,0.117,,030625,,,A*65'
}

nmea_sentences_active_2 = {
    "GPVTG": '$GPVTG,,,,,,,,,N*30 $GPGGA,202233.00,,,,,0,00,99.99,,,,,,*64',
    "GPGSA": '$GPGSA,A,3,30,20,13,14,11,,,,,,,,4.17,3.14,2.74*01',
    "GPGSV_1": '$GPGSV,4,1,14,05,68,261,,07,37,058,,08,02,052,,09,02,096,*70',
    "GPGSV_2": '$GPGSV,4,2,14,11,04,201,,13,57,276,,14,28,138,29,15,21,283,*7E',
    "GPGSV_3": '$GPGSV,4,3,14,18,18,322,,20,53,190,28,21,56,171,,22,15,152,*7B',
    "GPGSV_4":'$GPGSV,4,4,14,27,04,019,,30,69,085,*74',
    "GPGLL": '$GPGLL,,,,,202233.00,V,N*48',
    "GPRMC": '$GPRMC,200518.00,A,5212.77195,N,00212.27083,W,4.000,,030625,,,A*6C'
}

nmea_sentences_void = {
    "GPVTG" :"$GPVTG,,,,,,,,,N*30 $GPGGA,202233.00,,,,,0,00,99.99,,,,,,*64",
    "GPGSA" :"$GPGSA,A,1,,,,,,,,,,,,,99.99,99.99,99.99*30 ",
    "GPGSV_1" :"$GPGSV,4,1,14,05,68,261,,07,37,058,,08,02,052,,09,02,096,*70", 
    "GPGSV_2" :"$GPGSV,4,2,14,11,04,201,,13,57,276,,14,28,138,29,15,21,283,*7E", 
    "GPGSV_3" :"$GPGSV,4,3,14,18,18,322,,20,53,190,28,21,56,171,,22,15,152,*7B", 
    "GPGSV_4" :"$GPGSV,4,4,14,27,04,019,,30,69,085,*74", 
    "GPGLL" :"$GPGLL,,,,,202233.00,V,N*48", 
    "GPRMC" :"$GPRMC,200520.00,V,,,,,,,090625,,,N*70"
}

nmea_sentences_active_3 = {
    "GPVTG": '$GPVTG,,,,,,,,,N*30 $GPGGA,202233.00,,,,,0,00,99.99,,,,,,*64',
    "GPGSA": '$GPGSA,A,3,30,20,13,14,11,,,,,,,,4.17,3.14,2.74*01',
    "GPGSV_1": '$GPGSV,4,1,14,05,68,261,,07,37,058,,08,02,052,,09,02,096,*70',
    "GPGSV_2": '$GPGSV,4,2,14,11,04,201,,13,57,276,,14,28,138,29,15,21,283,*7E',
    "GPGSV_3": '$GPGSV,4,3,14,18,18,322,,20,53,190,28,21,56,171,,22,15,152,*7B',
    "GPGSV_4":'$GPGSV,4,4,14,27,04,019,,30,69,085,*74',
    "GPGLL": '$GPGLL,,,,,202233.00,V,N*48',
    "GPRMC": '$GPRMC,200519.00,A,5213.77306,N,00211.27194,W,4.000,,030625,,,A*60'
}

from src.kelder_api.components.gps_new.interface import GPSInterface
from src.kelder_api.configuration.settings import Settings
from src.kelder_api.components.redis_client.redis_client import RedisClient
import datetime

redis_client = RedisClient()
gps_interface = GPSInterface(redis_client)

Add in course over ground code:

In [2]:
import math
import numpy as np

def bearing_degrees(lat1, lon1, lat2, lon2):
    """Calculate initial bearing (forward azimuth) from point 1 to point 2"""
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    
    dlon = lon2 - lon1
    
    y = math.sin(dlon) * math.cos(lat2)
    x = math.cos(lat1) * math.sin(lat2) - math.sin(lat1) * math.cos(lat2) * math.cos(dlon)
    
    bearing = math.atan2(y, x)
    
    # Convert to degrees and normalize to 0-360
    bearing = math.degrees(bearing)
    bearing = (bearing + 360) % 360
    
    return bearing

def average_bearing(bearings):
    """Average bearings using vector method"""
    bearings_rad = np.radians(bearings)
    
    # Convert to unit vectors
    x = np.mean(np.cos(bearings_rad))
    y = np.mean(np.sin(bearings_rad))
    
    # Get average bearing
    avg_bearing = np.degrees(np.arctan2(y, x))
    return (avg_bearing + 360) % 360

Testing

In [3]:
from src.kelder_api.components.velocity.service import VelocityCalculator

# Clear the redis GPS set
async with redis_client.get_connection() as redis:
    size = await redis.zcard("sensor:ts:GPS")
    # print(f"Redis set size: {size}")
    await redis.zremrangebyrank("sensor:ts:GPS", 0, -1)

# gps_latest = await gps_interface.read_gps_all_history()
# for gps_measurement in gps_latest:
#     print(gps_measurement)

await gps_interface.stream_serial_data(list(nmea_sentences_active_1.values()))
# await gps_interface.stream_serial_data(list(nmea_sentences_active_2.values()))
# await gps_interface.stream_serial_data(list(nmea_sentences_active_3.values()))
await gps_interface.stream_serial_data(list(nmea_sentences_void.values()))

now = datetime.datetime(2025, 6, 3, 20, 5, 19)
velocity_calculator = VelocityCalculator(gps_interface=gps_interface, redis_client=redis_client)
await velocity_calculator.calculate_gps_velocity(now)

Insufficient GPS history for a velocity measurement, length: 1


In [4]:
await velocity_calculator.read_velocity_latest(active=False)

GPSVelocity(timestamp=datetime.datetime(2025, 6, 3, 20, 5, 19), speed_over_ground=None, number_of_measurements=1)

In [6]:
await velocity_calculator.read_velocity_all(active=True)

[GPSVelocity(timestamp=datetime.datetime(2025, 6, 3, 20, 5, 19), speed_over_ground=2.1745691217861074, number_of_measurements=3)]